In [ ]:
!pip install torchvision
!pip install fvcore umap-learn scikit-video opencv-python-headless matplotlib seaborn tqdm scikit-learn imblearn albumentations

In [ ]:
import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.parallel # For DataParallel
import torch.hub # Use torch.hub for model loading
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, auc, balanced_accuracy_score,
    matthews_corrcoef, silhouette_score
)
from scipy.stats import kendalltau, spearmanr, entropy, wasserstein_distance
from scipy.linalg import sqrtm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from PIL import Image
import warnings
import umap
from collections import Counter # For JS Divergence in evaluate

warnings.filterwarnings("ignore")

# Define the VideoFeatureExtractor class using R(2+1)D via torch.hub
class VideoFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super(VideoFeatureExtractor, self).__init__()
        # Load R(2+1)D-50 model from PyTorchVideo via torch.hub
        self.model = torch.hub.load('facebookresearch/pytorchvideo:main', 'r2plus1d_r50', pretrained=pretrained)
        # Replace the final classification block (head) with Identity to get features
        # The head typically includes AvgPool and Linear layer.
        # By replacing it, we get the output of the convolutional backbone.
        if hasattr(self.model, 'blocks') and len(self.model.blocks) > 0:
             # Assuming the head is the last item in 'blocks'
            self.model.blocks[-1] = nn.Identity()
        elif hasattr(self.model, 'head'): # Common attribute name for head
            self.model.head = nn.Identity()
        else:
            # Fallback: Try to identify and remove the last linear layer if possible,
            # or warn the user that automatic head removal might not be perfect.
            # This part might need adjustment based on the exact model structure from torch.hub.
            children = list(self.model.children())
            if isinstance(children[-1], nn.Linear):
                self.model = nn.Sequential(*children[:-1])
                print("Warning: Removed the last nn.Linear layer. Ensure this is the correct feature layer.")
            elif hasattr(children[-1], 'fc') and isinstance(getattr(children[-1], 'fc'), nn.Linear): # For ResNet-like structures
                 setattr(children[-1], 'fc', nn.Identity())
                 self.model = nn.Sequential(*children)
                 print("Warning: Replaced fc layer in the last block with Identity. Ensure this is the correct feature layer.")
            else:
                print("Warning: Could not automatically remove the classification head. Features might be post-activation values from the original head.")


    def forward(self, x):
        # Input x: (batch_size, C, T, H, W), e.g., (batch_size, 3, 32, 224, 224)
        features = self.model(x)
        # If the output is still 5D (e.g. B, C_feat, T_feat, H_feat, W_feat),
        # apply global average pooling over temporal and spatial dimensions.
        if features.dim() > 2 and features.shape[-1] > 1 and features.shape[-2] > 1 and features.shape[-3] > 1 : # Check if pooling is needed
            features = torch.mean(features, dim=[2, 3, 4])
        elif features.dim() == 5: # If it's BxCx1x1x1 (e.g. after some form of pooling in the model)
            features = features.squeeze(-1).squeeze(-1).squeeze(-1)
        return features # Output: (batch_size, feature_dim)

# Modified VideoDataset for R(2+1)D model and Optical Flow
class VideoDataset(Dataset):
    def __init__(self, data_dir, label_encoder, fraction=1.0): # Removed transform argument, handled internally
        self.data_dir = data_dir
        self.label_encoder = label_encoder
        self.fraction = fraction
        self.video_files = self._get_video_files()
        self.labels = [self.label_encoder.transform([class_name])[0] for _, class_name in self.video_files]

        # Augmentations for PIL Images (applied before ToTensor)
        self.pil_augmentations = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(10),
            transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
            transforms.RandomPerspective(distortion_scale=0.1, p=0.2)
        ])
        # Transform to convert PIL to Tensor and normalize
        self.tensor_transform = transforms.Compose([
            transforms.ToTensor(), # Converts PIL (H,W,C) [0-255] to Tensor (C,H,W) [0-1]
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet stats
        ])
        
        # Precompute optical flow variances (from original script)
        self.flow_variances = self._compute_all_flow_variances()
    
    def _get_video_files(self):
        video_files = []
        for class_name in os.listdir(self.data_dir):
            class_path = os.path.join(self.data_dir, class_name)
            if os.path.isdir(class_path):
                files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.mp4')]
                num_samples = max(1, int(len(files) * self.fraction))
                sampled_files = random.sample(files, num_samples) if len(files) > num_samples else files
                video_files.extend([(f, class_name) for f in sampled_files])
        return video_files
    
    def _compute_flow_variance(self, video_path): # For Optical Flow (original script)
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            cap.release()
            return 0
        # Use 16 frames for optical flow as in original
        indices = np.linspace(0, total_frames - 1, min(16, total_frames), dtype=int)
        frames_flow = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frames_flow.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
        cap.release()
        
        if len(frames_flow) < 2:
            return 0
            
        flows = []
        for i in range(len(frames_flow)-1):
            flow = cv2.calcOpticalFlowFarneback(frames_flow[i], frames_flow[i+1], None, 
                                                0.5, 3, 15, 3, 5, 1.2, 0)
            if flow is not None:
                mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
                flows.append(mag)
        
        if not flows:
            return 0
        return np.mean([np.var(f) for f in flows if f is not None and f.size > 0])
    
    def _compute_all_flow_variances(self): # For Optical Flow (original script)
        variances = []
        for video_path, _ in tqdm(self.video_files, desc="Computing Flow Variance"):
            variances.append(self._compute_flow_variance(video_path))
        return np.array(variances)
    
    def __len__(self):
        return len(self.video_files)
    
    def __getitem__(self, idx):
        video_path, class_name = self.video_files[idx]
        # Frame extraction for R(2+1)D model (32 frames)
        pil_frames = self._extract_frames_for_model(video_path)
        
        augmented_tensor_frames = []
        for frame_pil in pil_frames:
            # Apply PIL-based augmentations
            augmented_pil = self.pil_augmentations(frame_pil)
            # Convert to tensor and normalize
            tensor_frame = self.tensor_transform(augmented_pil)
            augmented_tensor_frames.append(tensor_frame)
            
        # Stack frames: list of (C,H,W) -> (T,C,H,W) -> (C,T,H,W)
        frames_tensor = torch.stack(augmented_tensor_frames).permute(1, 0, 2, 3)
        label = self.label_encoder.transform([class_name])[0]
        return frames_tensor, label
    
    def _extract_frames_for_model(self, video_path): # For R(2+1)D model
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames_pil = []
        
        if total_frames == 0: # Handle corrupted or empty video
            cap.release()
            # Return 32 blank PIL images
            return [Image.new('RGB', (224, 224)) for _ in range(32)]

        # Extract 32 frames for the R(2+1)D model
        indices = np.linspace(0, total_frames - 1, min(32, total_frames), dtype=int)
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (224, 224))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames_pil.append(Image.fromarray(frame))
            else: # If a frame can't be read, append last good frame or blank
                if frames_pil:
                    frames_pil.append(frames_pil[-1].copy())
                else:
                    frames_pil.append(Image.new('RGB', (224, 224)))

        cap.release()
        
        # Pad if fewer than 32 frames were extracted
        while len(frames_pil) < 32:
            if frames_pil:
                frames_pil.append(frames_pil[-1].copy())
            else: # Should not happen if total_frames > 0 logic is correct
                frames_pil.append(Image.new('RGB', (224, 224)))
        return frames_pil[:32] # Ensure exactly 32 frames


# Hybrid Classifier Class (modified fit and predict from second script, updated predict_proba)
class HybridClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, C=1.0, gamma='scale', kernel='rbf', weight=0.5):
        self.n_neighbors = n_neighbors
        self.C = C
        self.gamma = gamma
        self.kernel = kernel
        self.weight = weight
        self.knn = None
        self.svm = None
        self.classes_ = None

    def fit(self, X, y):
        self.knn = KNeighborsClassifier(n_neighbors=self.n_neighbors).fit(X, y)
        self.svm = SVC(C=self.C, gamma=self.gamma, kernel=self.kernel, probability=True).fit(X, y)
        self.classes_ = self.svm.classes_ # Store classes seen by SVM
        return self

    def predict(self, X):
        svm_probs = self.svm.predict_proba(X)
        
        _, knn_indices = self.knn.kneighbors(X)
        neighbor_labels = self.knn._y[knn_indices] # KNN's training labels for indices

        knn_contrib_list = []
        for single_sample_neighbor_labels in neighbor_labels:
            counts = np.bincount(single_sample_neighbor_labels, minlength=len(self.classes_))
            knn_contrib_list.append(counts)
        
        knn_contrib = np.array(knn_contrib_list) / self.n_neighbors
        
        hybrid_probs = (1 - self.weight) * svm_probs + self.weight * knn_contrib
        return self.classes_[np.argmax(hybrid_probs, axis=1)] # Return actual class labels

    def predict_proba(self, X):
        svm_probs = self.svm.predict_proba(X)
        
        _, knn_indices = self.knn.kneighbors(X)
        neighbor_labels = self.knn._y[knn_indices]

        if self.classes_ is None:
            raise ValueError("Classifier has not been fitted yet or classes_ not set.")

        knn_contrib_list = []
        for single_sample_neighbor_labels in neighbor_labels:
            counts = np.bincount(single_sample_neighbor_labels, minlength=len(self.classes_))
            knn_contrib_list.append(counts)
        
        knn_contrib = np.array(knn_contrib_list) / self.n_neighbors
        
        hybrid_probs = (1 - self.weight) * svm_probs + self.weight * knn_contrib
        return hybrid_probs


# Feature Extraction Function (updated with assert from second script)
def extract_features(model, loader, device='cuda'):
    model.eval()
    features_list, labels_list = [], []
    with torch.no_grad():
        for frames, lbls in tqdm(loader, desc="Extracting Features"):
            frames = frames.to(device)
            feats = model(frames)
            features_list.append(feats.cpu().numpy())
            labels_list.extend(lbls.numpy())
    
    if not features_list: # Handle empty loader case
        return np.array([]), np.array([])

    features_arr = np.vstack(features_list)
    assert features_arr.ndim == 2, f"Expected 2D features, got shape {features_arr.shape}"
    return features_arr, np.array(labels_list)


# Fréchet Video Distance Implementation (unchanged from first script)
def calculate_fvd(real_features, fake_features):
    if real_features.shape[0] < 2 or fake_features.shape[0] < 2: # Covariance matrix needs at least 2 samples
        print("Warning: Not enough samples to compute FVD reliably.")
        return float('nan')
    mu1, sigma1 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = np.mean(fake_features, axis=0), np.cov(fake_features, rowvar=False)
    
    # Add small epsilon for numerical stability if sigma might be singular
    epsilon = 1e-6
    sigma1_reg = sigma1 + np.eye(sigma1.shape[0]) * epsilon
    sigma2_reg = sigma2 + np.eye(sigma2.shape[0]) * epsilon
    
    diff = np.sum((mu1 - mu2)**2)
    
    try:
        covmean = sqrtm(sigma1_reg.dot(sigma2_reg))
    except ValueError: # Handle cases where sqrtm might fail (e.g. singular matrix despite regularization)
        print("Warning: sqrtm computation failed for FVD. Using pseudo-inverse or returning NaN.")
        try:
            # Attempt with pseudo-inverse based approaches or simpler forms if direct sqrtm fails
            # This is a placeholder for more advanced handling if needed
            covmean = sqrtm(sigma1.dot(sigma2), check_finite=False) # Try without strict checks
        except:
            return float('nan')


    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff + np.trace(sigma1 + sigma2 - 2*covmean)
    return fid


# Evaluation Function with All Metrics (from first script, ensures train/test_dataset are available)
def evaluate(true_labels, pred_labels, label_encoder, train_features, test_features, 
             train_dataset_eval, test_dataset_eval, flow_variances, feature_importance=None): # Added dataset args
    metrics = {
        'F1': f1_score(true_labels, pred_labels, average='weighted', zero_division=0),
        'Precision': precision_score(true_labels, pred_labels, average='weighted', zero_division=0),
        'Recall': recall_score(true_labels, pred_labels, average='weighted', zero_division=0),
        'Accuracy': accuracy_score(true_labels, pred_labels),
        'Kendall': kendalltau(true_labels, pred_labels)[0],
        'Spearman': spearmanr(true_labels, pred_labels)[0],
        'Balanced Accuracy': balanced_accuracy_score(true_labels, pred_labels),
        'MCC': matthews_corrcoef(true_labels, pred_labels)
    }
    if len(np.unique(true_labels)) > 1 and len(test_features) > 1 : # Silhouette score needs >1 cluster and >1 sample
         metrics['Silhouette Score'] = silhouette_score(test_features, true_labels, metric='euclidean')
    else:
        metrics['Silhouette Score'] = 0

    print("\nEvaluation Results:")
    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")
    
    # Confusion Matrix
    plt.figure(figsize=(14, 10))
    cm = confusion_matrix(true_labels, pred_labels, labels=label_encoder.transform(label_encoder.classes_))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('Confusion Matrix')
    plt.savefig('confusion_matrix.png')
    plt.show()
    
    # Per-Class F1 Score
    per_class_f1 = f1_score(true_labels, pred_labels, average=None, labels=label_encoder.transform(label_encoder.classes_), zero_division=0)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=label_encoder.classes_, y=per_class_f1)
    plt.xticks(rotation=45, ha="right")
    plt.title('Per-Class F1 Score')
    plt.ylabel('F1 Score')
    plt.xlabel('Class')
    plt.tight_layout()
    plt.savefig('per_class_f1.png')
    plt.show()
    
    # Fréchet Video Distance
    if train_features.size > 0 and test_features.size > 0:
        fvd = calculate_fvd(train_features, test_features)
        print(f"\nFréchet Video Distance: {fvd:.4f}")
    else:
        print("\nFréchet Video Distance: Not computed (empty features).")

    # Wasserstein Distance
    if train_features.size > 0 and test_features.size > 0 and train_features.shape[1] == test_features.shape[1]:
        wasserstein_dist = 0
        for i in range(train_features.shape[1]):
            wasserstein_dist += wasserstein_distance(train_features[:,i], test_features[:,i])
        wasserstein_dist /= train_features.shape[1]
        print(f"Wasserstein Distance (avg per feature dim): {wasserstein_dist:.4f}")
    else:
        print("Wasserstein Distance: Not computed (empty or mismatched features).")
        
    # Optical Flow Variance
    if flow_variances['train'].size > 0 or flow_variances['test'].size > 0:
        plt.figure(figsize=(10, 6))
        if flow_variances['train'].size > 0:
            sns.histplot(flow_variances['train'], kde=True, label='Train', alpha=0.5, stat="density")
        if flow_variances['test'].size > 0:
            sns.histplot(flow_variances['test'], kde=True, label='Test', alpha=0.5, stat="density")
        plt.title('Optical Flow Variance Distribution')
        plt.xlabel('Variance')
        plt.ylabel('Density')
        plt.legend()
        plt.savefig('optical_flow_variance.png')
        plt.show()
    
    # Feature Importance (PCA)
    if feature_importance is not None and feature_importance.size > 0:
        plt.figure(figsize=(12, 6))
        sns.barplot(x=np.arange(feature_importance.shape[0]), y=feature_importance)
        plt.title('Feature Importance (PCA-based)')
        plt.xlabel('Principal Components')
        plt.ylabel('Importance')
        plt.savefig('feature_importance.png')
        plt.show()
    
    # t-SNE & UMAP Visualizations
    if train_features.size > 0 and test_features.size > 0:
        combined_features = np.vstack([train_features, test_features])
        dataset_labels_viz = ['Train'] * len(train_features) + ['Test'] * len(test_features)
        
        # t-SNE
        perplexity_val = min(30, len(combined_features) -1 ) # Perplexity must be less than n_samples
        if perplexity_val > 0 :
            tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity_val, n_iter=300) # Reduced n_iter for speed
            reduced_tsne = tsne.fit_transform(combined_features)
            plt.figure(figsize=(10, 6))
            sns.scatterplot(x=reduced_tsne[:, 0], y=reduced_tsne[:, 1], hue=dataset_labels_viz, alpha=0.7, palette='Set1', s=50)
            plt.title('t-SNE Visualization: Train vs Test Feature Space')
            plt.savefig('tsne_dataset_comparison.png')
            plt.show()
        else:
            print("Skipping t-SNE: Not enough samples for perplexity.")

        # UMAP
        if len(combined_features) > 5: # UMAP default n_neighbors is 15, ensure enough samples
            reducer = umap.UMAP(random_state=42, n_neighbors=min(15, len(combined_features)-1), min_dist=0.1)
            reduced_umap = reducer.fit_transform(combined_features)
            plt.figure(figsize=(10, 6))
            sns.scatterplot(x=reduced_umap[:, 0], y=reduced_umap[:, 1], hue=dataset_labels_viz, alpha=0.7, palette='Set2', s=50)
            plt.title('UMAP Visualization: Train vs Test Feature Space')
            plt.savefig('umap_dataset_comparison.png')
            plt.show()
        else:
            print("Skipping UMAP: Not enough samples.")

    # Class Distribution Divergence (Jensen-Shannon Divergence)
    train_counter = Counter(train_dataset_eval.labels)
    test_counter = Counter(test_dataset_eval.labels)
    all_classes_indices = label_encoder.transform(label_encoder.classes_) # Get numeric indices for all classes

    train_dist = np.array([train_counter.get(i, 0) for i in all_classes_indices])
    test_dist = np.array([test_counter.get(i, 0) for i in all_classes_indices])
    
    if train_dist.sum() > 0 and test_dist.sum() > 0 :
        train_dist_norm = train_dist / train_dist.sum()
        test_dist_norm = test_dist / test_dist.sum()
        m = 0.5 * (train_dist_norm + test_dist_norm)
        # Add epsilon to avoid log(0) in entropy calculation
        epsilon_jsd = 1e-9
        js_div = 0.5 * (entropy(train_dist_norm + epsilon_jsd, m + epsilon_jsd) + entropy(test_dist_norm + epsilon_jsd, m + epsilon_jsd))
        print(f"\nJensen-Shannon Divergence between train and test label distributions: {js_div:.4f}")
    else:
        print("\nJensen-Shannon Divergence: Not computed (empty train or test distributions).")


# Grid Search Helper (from second script)
def build_grid(best_params):
    grid = {}
    best_n = best_params['classifier__n_neighbors']
    grid['classifier__n_neighbors'] = sorted(list(set([max(3, best_n - 2), best_n, best_n + 2]))) # Ensure unique sorted, min 3
    
    best_C = best_params['classifier__C']
    grid['classifier__C'] = sorted(list(set([best_C / 2, best_C, best_C * 2])))
    
    best_gamma = best_params['classifier__gamma']
    if isinstance(best_gamma, str): # e.g. 'scale', 'auto'
        grid['classifier__gamma'] = sorted(list(set([best_gamma, 'auto' if best_gamma != 'auto' else 'scale'])))
    else: # float
        grid['classifier__gamma'] = sorted(list(set([best_gamma * 0.5, best_gamma, best_gamma * 2])))
        
    best_kernel = best_params['classifier__kernel']
    grid['classifier__kernel'] = sorted(list(set([best_kernel, 'linear' if best_kernel != 'linear' else 'rbf'])))
    
    best_weight = best_params['classifier__weight']
    grid['classifier__weight'] = sorted(list(set([round(max(0.0, best_weight - 0.1),2), round(best_weight,2), round(min(1.0, best_weight + 0.1),2)])))
    return grid


# Main Function with R(2+1)D Model and Optical Flow Integration
def main():
    global train_dataset, test_dataset, hybrid_model # For potential access outside
    # Setup paths
    train_dir = '/kaggle/input/sum-yt/Youtube summary/Train' # Example path
    test_dir = '/kaggle/input/sum-yt/Youtube summary/Test'   # Example path

    # Check if paths exist
    if not os.path.isdir(train_dir):
        print(f"Train directory not found: {train_dir}. Please check the path.")
        # Fallback to a dummy structure if on a system without the data, for testing script logic
        print("Creating dummy data directories for testing purposes.")
        os.makedirs(os.path.join(train_dir, "classA"), exist_ok=True)
        os.makedirs(os.path.join(train_dir, "classB"), exist_ok=True)
        os.makedirs(os.path.join(test_dir, "classA"), exist_ok=True)
        os.makedirs(os.path.join(test_dir, "classB"), exist_ok=True)
        # Create a few dummy .mp4 files (0-byte is fine for dataset listing)
        for i in range(2):
            open(os.path.join(train_dir, "classA", f"dummy_train_A_{i}.mp4"), 'a').close()
            open(os.path.join(train_dir, "classB", f"dummy_train_B_{i}.mp4"), 'a').close()
            open(os.path.join(test_dir, "classA", f"dummy_test_A_{i}.mp4"), 'a').close()
            open(os.path.join(test_dir, "classB", f"dummy_test_B_{i}.mp4"), 'a').close()


    # Label encoding
    label_encoder = LabelEncoder()
    # Ensure class_dirs are only actual directories and sort them
    class_dirs = sorted([d for d in os.listdir(train_dir) 
                         if os.path.isdir(os.path.join(train_dir, d))])
    if not class_dirs:
        print(f"No subdirectories found in {train_dir}. Ensure it's populated with class folders.")
        return
    label_encoder.fit(class_dirs)
    
    # Create datasets (no external transform needed, it's handled internally by VideoDataset)
    print("Initializing Train Dataset...")
    train_dataset = VideoDataset(train_dir, label_encoder, fraction=1.0) # Using 100% of data
    print("Initializing Test Dataset...")
    test_dataset = VideoDataset(test_dir, label_encoder, fraction=1.0)
    
    if len(train_dataset) == 0 or len(test_dataset) == 0:
        print("Train or Test dataset is empty. Cannot proceed.")
        return

    # Create data loaders (num_workers from second script)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, # Reduced batch size for memory
                              num_workers=2, pin_memory=True, prefetch_factor=2) # Reduced num_workers
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False,
                             num_workers=2, pin_memory=True, prefetch_factor=2)
    
    # Load R(2+1)D feature extractor model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    model = VideoFeatureExtractor(pretrained=True)
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for feature extraction model!")
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Feature extraction
    print("Extracting training features...")
    train_features, train_labels = extract_features(model, train_loader, device)
    print("Extracting testing features...")
    test_features, test_labels = extract_features(model, test_loader, device)

    if train_features.size == 0 or test_features.size == 0:
        print("Feature extraction resulted in empty arrays. Cannot proceed with training/evaluation.")
        return

    # Build pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', HybridClassifier())
    ])
    
    # Hyperparameter search (ranges and CV settings from second script)
    param_dist_random = {
        'classifier__n_neighbors': list(range(3, 31, 2)), # Reduced upper limit from 51 for speed
        'classifier__C': np.logspace(-3, 3, 7), # Slightly reduced range/steps
        'classifier__gamma': ['scale', 'auto'] + list(np.logspace(-3, 2, 6)), # Slightly reduced
        'classifier__kernel': ['rbf', 'linear', 'poly'], # Removed sigmoid as it can be unstable
        'classifier__weight': np.linspace(0.0, 1.0, 6) # Reduced steps
    }
    
    # Stratified K-Fold for cross-validation
    # Using min(5, number_of_samples_in_smallest_class) if classes are imbalanced.
    # For simplicity here, using a fixed number or small CV if dataset is small.
    n_splits_cv = 2 # Reduced CV folds for speed
    try:
        # Check if stratified split is possible
        min_class_count_train = np.min(np.bincount(train_labels)) if len(train_labels) > 0 else 0
        if min_class_count_train < n_splits_cv and min_class_count_train > 0:
            n_splits_cv = min_class_count_train
            print(f"Warning: Reduced CV folds to {n_splits_cv} due to small class size in training data.")
        elif min_class_count_train == 0 and len(train_labels) > 0 : # Single class in train_labels
             n_splits_cv = StratifiedKFold(n_splits=n_splits_cv, shuffle=True, random_state=42) # will likely fail or warn
             print(f"Warning: Training data might have only one class or be too small for stratified {n_splits_cv}-fold CV.")
        elif len(train_labels) == 0:
            print("Training labels are empty. Skipping hyperparameter search.")
            return
        
        cv_strategy = StratifiedKFold(n_splits=n_splits_cv, shuffle=True, random_state=42)

    except ValueError: # Fallback if StratifiedKFold cannot be formed
        print("Could not form StratifiedKFold, falling back to standard KFold for CV.")
        from sklearn.model_selection import KFold
        cv_strategy = KFold(n_splits=n_splits_cv, shuffle=True, random_state=42)


    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist_random,
        n_iter=5, # Reduced n_iter from 10 for faster execution
        cv=cv_strategy,
        scoring='f1_weighted',
        n_jobs=-1, # Use all available cores
        verbose=1,
        random_state=42
    )
    print("Starting Randomized Search for hyperparameters...")
    random_search.fit(train_features, train_labels)
    print("\nBest Random Search Params:")
    for key, value in random_search.best_params_.items():
        print(f"{key}: {value}")
    
    param_grid_refined = build_grid(random_search.best_params_)
    print("\nRefined Grid for Grid Search:")
    for key, value in param_grid_refined.items():
        print(f"{key}: {value}")

    grid_search = GridSearchCV(
        pipeline,
        param_grid=param_grid_refined,
        cv=cv_strategy, # Use same CV strategy
        scoring='f1_weighted',
        n_jobs=-1, # Use all available cores
        verbose=1
    )
    print("Starting Grid Search for hyperparameters...")
    grid_search.fit(train_features, train_labels)
    print("\nBest Grid Search Params:")
    for key, value in grid_search.best_params_.items():
        print(f"{key}: {value}")
    
    hybrid_model = grid_search.best_estimator_
    predictions = hybrid_model.predict(test_features)
    
    # Calculate feature importance using PCA (from first script)
    feature_importance_pca = None
    if train_features.shape[1] > 1: # PCA needs more than 1 feature
        pca_components = min(train_features.shape[1], 10) # Cap at 10 or num_features
        if pca_components > 0:
            pca = PCA(n_components=pca_components)
            pca.fit(train_features)
            # A simple way to get importance: sum of absolute values of components for each original feature
            # This is a simplification. True "feature importance" from PCA components is complex.
            # More often, explained_variance_ratio_ per component is shown.
            # For this context, let's use explained_variance_ratio_ as "importance" of the component.
            feature_importance_pca = pca.explained_variance_ratio_ 
        else:
            print("Not enough features for PCA based importance.")
    else:
        print("Not enough features for PCA based importance (need >1).")

    # Collect flow variances (from first script)
    flow_variances_data = {
        'train': train_dataset.flow_variances,
        'test': test_dataset.flow_variances
    }
    
    # Generate all evaluation plots with new metrics
    evaluate(test_labels, predictions, label_encoder, 
             train_features, test_features, 
             train_dataset, test_dataset, # Pass datasets for JSD
             flow_variances_data, feature_importance_pca)

if __name__ == "__main__":
    main()